In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Sintetizarea operațiilor unitare

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
```
</details>
O operație unitară descrie o modificare ce conservă norma unui sistem cuantic.
Pentru $n$ qubiți, această modificare este descrisă de o matrice complexă de dimensiune $2^n \times 2^n$, notată $U$, al cărei adjunct este egal cu inversa sa, adică $U^\dagger U = \mathbb{1}$.

Sintetizarea unor operații unitare specifice într-un set de Gate-uri cuantice este o sarcină fundamentală, folosită, de exemplu, în proiectarea și aplicarea algoritmilor cuantici sau în compilarea Circuit-urilor cuantice.

Deși sinteza eficientă este posibilă pentru anumite clase de unitare — precum cele compuse din Gate-uri Clifford sau cu structură de produs tensorial — majoritatea unitarelor nu se încadrează în aceste categorii.
Pentru matricele unitare generale, sinteza este o sarcină complexă, ale cărei costuri de calcul cresc exponențial cu numărul de qubiți.
Prin urmare, dacă știi o descompunere eficientă pentru unitara pe care dorești să o implementezi, aceasta va fi probabil mai bună decât o sinteză generală.

> **Note:** Dacă nu este disponibilă nicio descompunere, Qiskit SDK îți pune la dispoziție instrumentele necesare pentru a găsi una.
>     Reține, totuși, că aceasta generează în general Circuit-uri adânci, care pot fi nepotrivite pentru rularea pe calculatoare cuantice zgomotoase.

In [1]:
import numpy as np
from qiskit import QuantumCircuit

U = 0.5 * np.array(
    [[1, 1, 1, 1], [-1, 1, -1, 1], [-1, -1, 1, 1], [-1, 1, 1, -1]]
)

circuit = QuantumCircuit(2)
circuit.unitary(U, circuit.qubits)

## Re-synthesis for circuit optimization

Sometimes it is beneficial to re-synthesize a long series of single- and two-qubit gates, if the length can be reduced. For example, the following circuit uses three two-qubit gates.

In [2]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit

qreg_q = QuantumRegister(2, "q")
creg_c = ClassicalRegister(4, "c")
circuit = QuantumCircuit(qreg_q, creg_c)

circuit.h(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.sx(qreg_q[1])
circuit.cz(qreg_q[0], qreg_q[1])
circuit.x(qreg_q[1])
circuit.x(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.h(qreg_q[0])
circuit.draw("mpl")

<Image src="../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg" alt="Output of the previous code cell" />

## Re-sintetizarea pentru optimizarea Circuit-urilor
Uneori este avantajos să re-sintetizezi o serie lungă de Gate-uri pe unul sau doi qubiți, dacă lungimea poate fi redusă. De exemplu, următorul Circuit folosește trei Gate-uri pe doi qubiți.

In [3]:
from qiskit.quantum_info import Operator

# compute unitary matrix of circuit
U = Operator(circuit)

# re-synthesize
better_circuit = QuantumCircuit(2)
better_circuit.unitary(U, range(2))
better_circuit.decompose().draw()

global phase: 6.2071
      ┌───────────────┐         ┌────────────────┐ 
q_0: ─┤ U(π/2,π/2,-π) ├────■────┤ U(π/2,-π,-π/2) ├─
     ┌┴───────────────┴─┐┌─┴─┐┌─┴────────────────┴┐
q_1: ┤ U(1.7229,π/2,-π) ├┤ X ├┤ U(π/2,0.15207,-π) ├
     └──────────────────┘└───┘└───────────────────┘

![Output of the previous code cell](../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg)

Cu toate acestea, după re-sintetizare folosind codul de mai jos, sunt necesare doar un singur Gate CX. (Aici folosim metoda `QuantumCircuit.decompose()` pentru a vizualiza mai bine Gate-urile utilizate la re-sintetizarea unitarei.)